# Ewaluacja Testu A/B (Wymaganie b.iii)
**Analiza logów z mikroserwisu produkcyjnego**

Poniższy skrypt wczytuje logi wygenerowane przez nasz mikroserwis (FastAPI) w trakcie trwania testu A/B. 
Ruch z zapytań HTTP (np. via `curl`) był losowo rozdzielany pomiędzy:
* **Model A:** Isolation Forest (Baseline)
* **Model B:** Random Forest (Ostateczny model docelowy)

Cel: Weryfikacja, jak często poszczególne modele flagują oferty jako anomalie (zawyżone ceny) i czy różnica w ich zachowaniu jest istotna statystycznie (Test Chi-Kwadrat).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
import os

plt.style.use('seaborn-v0_8-whitegrid')

LOG_FILE = "ab_test_logs.csv"

if os.path.exists(LOG_FILE):
    df_logs = pd.read_csv(LOG_FILE)
    print(f"Wczytano {len(df_logs)} zdarzeń z logów.")
else:
    print("Brak pliku z logami. Uruchom najpierw mikroserwis i wyślij zapytania via curl.")
    print("Generowanie przykładowych danych do demonstracji skryptu...")
    df_logs = pd.DataFrame({
        "model_assigned": ["Model_A_IsolationForest"]*50 + ["Model_B_RandomForest"]*50,
        "prediction_result": [1]*30 + [0]*20 + [1]*15 + [0]*35
    })

In [ ]:
# ============================================================
# BASIC STATISTICS AND CHARTS
# ============================================================

traffic_split = df_logs['model_assigned'].value_counts(normalize=True) * 100
print("Podział ruchu w teście A/B (%):")
print(traffic_split)
print("\n")

conversion_rates = df_logs.groupby('model_assigned')['prediction_result'].mean() * 100
print("Odsetek ofert oflagowanych jako anomalia cenowa:")
print(conversion_rates)

plt.figure(figsize=(8, 5))
sns.barplot(x=conversion_rates.index, y=conversion_rates.values, palette="Set2")
plt.title("Test A/B: Odsetek wykrytych anomalii (Model A vs Model B)")
plt.ylabel("% Oflagowanych Ofert (Zawyżona cena)")
plt.xlabel("Model")
plt.ylim(0, 100)
plt.show()

## Weryfikacja Statystyczna (Test Chi-Kwadrat)
Aby sprawdzić, czy różnica w odsetku wykrywanych anomalii między modelem bazowym a docelowym jest wynikiem przypadku, czy rzeczywistej różnicy w skuteczności, przeprowadzamy test niezależności $\chi^2$ (Chi-Square) dla tabeli kontyngencji.

In [ ]:
# ============================================================
# CHI-SQUARE STATISTICAL TEST
# ============================================================

contingency_table = pd.crosstab(df_logs['model_assigned'], df_logs['prediction_result'])

print("Tabela kontyngencji (Obserwacje):")
print(contingency_table)

chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print(f"\nWyniki testu Chi-Kwadrat:")
print(f"Chi2 Statistic: {chi2:.4f}")
print(f"P-value: {p_value:.4f}")

alpha = 0.05
if p_value < alpha:
    print("\n[Wniosek]: P-value < 0.05. Odrzucamy hipotezę zerową.")
    print("Istnieje STATYSTYCZNIE ISTOTNA różnica w tym, jak oba modele klasyfikują oferty.")
else:
    print("\n[Wniosek]: P-value >= 0.05. Brak podstaw do odrzucenia hipotezy zerowej.")
    print("Na obecnym etapie (przy tej próbie) modele nie różnią się statystycznie w odsetku oflagowanych ofert.")

## Podsumowanie i Wnioski Końcowe
Skrypt poprawnie zlicza i ewaluuje logi z mikroserwisu.
Zgodnie z wymogami testu A/B, narzędzie to pozwala w sposób ciągły monitorować działanie modeli na produkcji. W docelowym wdrożeniu biznesowym, oprócz weryfikacji samego faktu "oflagowania", test ten zostanie rozszerzony o mierzenie realnej konwersji (np. czy dany host po otrzymaniu alertu o zawyżonej cenie faktycznie ją obniżył).

Ostatecznym zwycięzcą zostaje **Model B (Random Forest)**, który na podstawie ewaluacji z Notatnika 03 cechuje się wyższą swoistością (brak fałszywych alarmów denerwujących klientów) przy zachowaniu odpowiedniej czułości narzuconej w pliku *ML Canvas*.